In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu128
True


In [ ]:
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.5 MB/s eta 0:00:00


In [ ]:
import torch
import torchvision
import lpips
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

print(torch.__version__)
print(torchvision.__version__)
print("CUDA:", torch.cuda.is_available())

2.11.0+cu128
0.26.0+cu128
CUDA: True


In [ ]:
import os
import csv
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
import lpips
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
REAL_DIR = "/content/real"
FAKE_DIR = "/content/fastgan_fake"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

loss_fn_lpips = lpips.LPIPS(net='alex').to(device)

transform_lpips = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def load_image(path):
    img = Image.open(path).convert('RGB')
    return transform_lpips(img).unsqueeze(0).to(device)

real_files = [os.path.join(REAL_DIR, f) for f in os.listdir(REAL_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
fake_files = [os.path.join(FAKE_DIR, f) for f in os.listdir(FAKE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

print(f"Loaded {len(real_files)} real images and {len(fake_files)} fake images.")

min_mse_per_real = []
min_lpips_per_real = []

print("Running Quantitative Comparison...")
for real_path in tqdm(real_files):
    real_tensor = load_image(real_path)
    real_np = ((real_tensor.squeeze(0).cpu().numpy() * 0.5 + 0.5) * 255.0).astype(np.float32)

    current_real_mses = []
    current_real_lpips = []

    for fake_path in fake_files:
        fake_tensor = load_image(fake_path)

        # Compute LPIPS Distance
        with torch.no_grad():
            lpips_dist = loss_fn_lpips(real_tensor, fake_tensor).item()
            current_real_lpips.append(lpips_dist)

        # Compute MSE Distance
        fake_np = ((fake_tensor.squeeze(0).cpu().numpy() * 0.5 + 0.5) * 255.0).astype(np.float32)
        mse_dist = np.mean((real_np - fake_np) ** 2)
        current_real_mses.append(mse_dist)

    min_mse_per_real.append(min(current_real_mses))
    min_lpips_per_real.append(min(current_real_lpips))

print("\nGenerating and saving plots...")
image_numbers = list(range(len(min_mse_per_real)))

# MSE Plot
plt.figure(figsize=(7, 5))
plt.plot(image_numbers, min_mse_per_real, linewidth=1)
plt.title("MSE of nearest generated Image")
plt.xlabel("Image Number")
plt.ylabel("MSE")

stats_text_mse = f"Max: {max(min_mse_per_real):.2f}\nMin: {min(min_mse_per_real):.2f}\nAvg: {np.mean(min_mse_per_real):.2f}"
plt.text(0.95, 0.95, stats_text_mse, transform=plt.gca().transAxes,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig("mse_plot.png", dpi=300)
plt.close()
print("Saved: mse_plot.png")

# LPIPS Plot
plt.figure(figsize=(7, 5))
plt.plot(image_numbers, min_lpips_per_real, linewidth=1)
plt.title("LPIPS distance of nearest generated Image")
plt.xlabel("Image Number")
plt.ylabel("LPIPS Distance")

stats_text_lpips = f"Max: {max(min_lpips_per_real):.3f}\nMin: {min(min_lpips_per_real):.3f}\nAvg: {np.mean(min_lpips_per_real):.3f}"
plt.text(0.95, 0.95, stats_text_lpips, transform=plt.gca().transAxes,
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig("lpips_plot.png", dpi=300)
plt.close()
print("Saved: lpips_plot.png")

print("\nSaving data to CSV files...")

# MSE CSV
with open("mse_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Image_Number", "MSE"])
    for img_num, mse_val in zip(image_numbers, min_mse_per_real):
        writer.writerow([img_num, mse_val])
print("Saved: mse_data.csv")

# LPIPS CSV
with open("lpips_data.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Image_Number", "LPIPS"])
    for img_num, lpips_val in zip(image_numbers, min_lpips_per_real):
        writer.writerow([img_num, lpips_val])
print("Saved: lpips_data.csv")


Using device: cuda
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Loaded 953 real images and 476 fake images.
Running Quantitative Comparison...


100%|██████████| 953/953 [1:01:15<00:00,  3.86s/it]



Generating and saving plots...
Saved: mse_plot.png
Saved: lpips_plot.png

Saving data to CSV files...
Saved: mse_data.csv
Saved: lpips_data.csv

Done! Check your workspace folder for the generated plots and CSV files.
